# EDA Pipeline Example

This notebook shows how to run the reusable EDA pipeline on a pandas DataFrame and, when GeoPandas is installed, on a GeoDataFrame.

In [ ]:
from pathlib import Path
import os
import random

from import_libraries import *
from common_libraries.io_library import read_data
from eda_pipeline import run_eda
import webbrowser

# Global reproducibility seed
random_seed = 42
os.environ["PYTHONHASHSEED"] = str(random_seed)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(random_seed)
np.random.seed(random_seed)

## GeoDataFrame Example

In [ ]:
try:
    WORK_PATH = Path("C:/work_dir/volvi-2023-2024")
    PARCELS_PATH = WORK_PATH / "1_parcel_stats" / "satellite_parcel_time_stats.geoparquet"
    OUTPUT_PATH = WORK_PATH / "2_results_eda"
    COL_PARCEL_ID = "parcel_code"
    COL_TARGET = "label_gr"
    gdf = read_data(str(PARCELS_PATH))
    print(f"Initial columns in GeoDataFrame: {len(gdf.columns)}: {list(gdf.columns)}")

    extra_features = ["elevation_mean_m", "distance_to_nearest_lake_m", "nearest_lake_row_id", "GAIOIKANOT"]

    show_indices =  ("NDMI", "NDRE", "SAVI", "NDVI", "EVI", "NDWI",)
    show_statistics = ("_median",)
    base_columns = [x for x in gdf.columns if (any(stat in x for stat in show_statistics) and any(idx in x for idx in show_indices))]

    columns_to_display = [COL_PARCEL_ID, COL_TARGET] + base_columns + ["geometry"]
    print(f"\nColumns to display {len(columns_to_display)}: {columns_to_display}")

    display(gdf[columns_to_display].head())
    geo_result = run_eda(
        gdf[columns_to_display],
        output_dir=str(OUTPUT_PATH),
        target_task="classification",
        target_column=COL_TARGET,
        id_column=COL_PARCEL_ID,
        report_title="Sample GeoDataFrame EDA",
        feature_selection_max_features=None,
        max_features_per_plot=36,
    )
    html_path = geo_result["report_path"]
    proposed_features = geo_result["proposed_features"]
    annotated_gdf = geo_result["annotated_data"]
    display(geo_result["artifacts"]["feature_selection_proposal"])
    print(f"Annotated GeoParquet: {geo_result['saved_paths']['annotated_data']}")
    webbrowser.open(html_path.as_uri())
except ImportError as exc:
    print(f"GeoDataFrame example skipped: {exc}")

In [ ]:
import common_libraries.eda_report_class as eda_l

report = eda_l.EdaReport(gdf[columns_to_display], report_folder=os.path.join(str(OUTPUT_PATH), "geodataframe_eda2"), report_name="eda_report_sample")
report.create_eda_report()